# いろどりTTS 全自動生成

### 手順
1. **セル1** → Google Driveに接続（初回のみ許可が必要）
2. **セル2** → いろどりTTSをインストール（**初回だけ10分、2回目以降は30秒**）
3. **セル3** → blocks.txt と参照音声をアップロード
4. **セル4** → 全ブロック自動生成（放置でOK）
5. **セル5** → zipでダウンロード

> ランタイム → ランタイムのタイプを変更 → **T4 GPU** を選択してから実行

In [ ]:
# ============================================================
# セル1：Google Driveに接続
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive接続完了！')

In [ ]:
# ============================================================
# セル2：いろどりTTSのインストール
# 初回：Google Driveにインストール（約10分）
# 2回目以降：Drive上のキャッシュを使用（約30秒）
# ============================================================
import os
import subprocess

DRIVE_DIR = '/content/drive/MyDrive/IrodoriTTS'
LOCAL_DIR = '/content/Irodori-TTS'

os.makedirs(DRIVE_DIR, exist_ok=True)

if not os.path.exists(f'{DRIVE_DIR}/.git'):
    print('初回インストール中...（約10分）')
    !git clone https://github.com/Aratako/Irodori-TTS.git {DRIVE_DIR}
    !cd {DRIVE_DIR} && pip install -q uv && uv sync
    print('インストール完了！次回からは30秒で起動します')
else:
    print('キャッシュ済み - 高速起動中...')

# ローカルにシンボリックリンク
if not os.path.exists(LOCAL_DIR):
    !ln -s {DRIVE_DIR} {LOCAL_DIR}

os.chdir(LOCAL_DIR)
print(f'準備完了: {LOCAL_DIR}')

# GPU確認
!nvidia-smi | head -3

In [ ]:
# ============================================================
# セル3：blocks.txt と参照音声をアップロード
# ============================================================
from google.colab import files
import os

print('blocks.txt と 参照音声(.wav) を選択してください')
print('2つのファイルを同時に選択できます')
print()

uploaded = files.upload()

blocks_file = None
ref_wav = None

for filename in uploaded:
    if filename.endswith('.txt'):
        blocks_file = f'{LOCAL_DIR}/blocks.txt'
        with open(blocks_file, 'wb') as f:
            f.write(uploaded[filename])
        print(f'blocks.txt: OK')
    elif filename.endswith('.wav'):
        os.makedirs(f'{LOCAL_DIR}/ref', exist_ok=True)
        # ファイル名に(3)やスペースが入っていても関係なくreference.wavとして保存
        ref_wav = f'{LOCAL_DIR}/ref/reference.wav'
        with open(ref_wav, 'wb') as f:
            f.write(uploaded[filename])
        print(f'参照音声: OK (reference.wav として保存)')

if blocks_file:
    with open(blocks_file, 'r', encoding='utf-8') as f:
        content = f.read()
    blocks = [b.strip() for b in content.split('\n\n') if b.strip()]
    print(f'\nブロック数: {len(blocks)}')
    print(f'参照音声: {ref_wav or "なし（no-ref）"}')
else:
    print('⚠️ blocks.txt がアップロードされていません')

In [ ]:
# ============================================================
# セル4：全ブロック自動生成
# ============================================================
import subprocess, time, os
from pathlib import Path
from datetime import datetime

HF_CHECKPOINT = 'Aratako/Irodori-TTS-500M-v2'
SEED = 42

def normalize_text(text):
    text = text.replace('\u300c', '').replace('\u300d', '')
    text = text.replace('\u300e', '').replace('\u300f', '')
    text = text.replace('\u2026\u2026', '\u3002').replace('\u2026', '\u3002')
    text = text.replace('\u30fb\u30fb\u30fb', '\u3002')
    text = text.replace('\uff1f', '?').replace('\uff01', '!').replace('\uff5e', '~')
    text = text.replace('\u2015\u2015', '\u3002').replace('\u2015', '\u3001')
    stripped = text.rstrip()
    if stripped and stripped[-1] not in '\u3002\u3001!?~.':
        text = stripped + '\u3002'
    return text

def generate_one(text, output_wav, ref_wav=None, seed=42):
    text = normalize_text(text)
    cmd = ['uv', 'run', 'python', 'infer.py',
           '--hf-checkpoint', HF_CHECKPOINT,
           '--text', text,
           '--output-wav', output_wav,
           '--seed', str(seed),
           '--num-steps', '80',
           '--num-candidates', '1']
    if ref_wav and os.path.exists(ref_wav):
        cmd += ['--ref-wav', ref_wav]
    else:
        cmd += ['--no-ref']
    result = subprocess.run(cmd, cwd=LOCAL_DIR, capture_output=True, text=True, encoding='utf-8', errors='replace')
    return result.returncode == 0

with open(blocks_file, 'r', encoding='utf-8') as f:
    content = f.read()
blocks = [b.strip() for b in content.split('\n\n') if b.strip()]
total = len(blocks)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
out_dir = Path(f'{LOCAL_DIR}/gradio_outputs/{timestamp}')
out_dir.mkdir(parents=True, exist_ok=True)

print(f'=== いろどりTTS 全自動生成 ===')
print(f'ブロック数 : {total}')
print(f'参照音声   : {ref_wav or "なし"}')
print(f'出力先     : {out_dir}')
print()

failed = []
start_time = time.time()

for i, block in enumerate(blocks):
    block_num = i + 1
    out_wav = str(out_dir / f'block_{block_num:04d}.wav')
    if os.path.exists(out_wav):
        print(f'[{block_num:03d}/{total}] スキップ（既存）')
        continue
    preview = block[:35].replace('\n', ' ')
    print(f'[{block_num:03d}/{total}] 「{preview}...」', end=' ', flush=True)
    t0 = time.time()
    success = generate_one(block, out_wav, ref_wav, SEED)
    elapsed = time.time() - t0
    if success:
        print(f'✓ ({elapsed:.1f}秒)')
    else:
        print(f'✗ 失敗')
        failed.append(block_num)

total_time = time.time() - start_time
print(f'\n=== 完了 ===')
print(f'成功: {total - len(failed)} / {total}')
print(f'失敗: {failed if failed else "なし"}')
print(f'合計: {total_time/60:.1f}分')
print(f'出力先: {out_dir}')

In [ ]:
# ============================================================
# セル5：生成したwavをzipにまとめてダウンロード
# ============================================================
import zipfile
from google.colab import files
from pathlib import Path

wav_files = sorted(out_dir.glob('*.wav'))
zip_path = f'/content/output_{timestamp}.zip'

print(f'{len(wav_files)}個のwavをzip化中...')
with zipfile.ZipFile(zip_path, 'w') as zf:
    for wav in wav_files:
        zf.write(wav, wav.name)

print('ダウンロード開始...')
files.download(zip_path)